# DAY4 실습 04 — 카이제곱 동질성 검정

In [ ]:
%pip install -q koreanize-matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
from scipy import stats

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
data_folder = "/content/drive/MyDrive/HD-AI/data"
csv_path = os.path.join(data_folder, "ai4i2020.csv")

df = pd.read_csv(csv_path)

data = df.copy()

Mounted at /content/drive


## 데이터와 질문

데이터 파일 : `ai4i2020.csv`
출처 : UCI Machine Learning Repository - [AI4I 2020 Predictive Maintenance Dataset](https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset)

| 열 | 뜻 |
|---|---|
| `Type` | 제품 유형 (L : 저품질, M : 중품질, H : 고품질) |
| `Machine failure` | 고장 여부 (1 : 고장, 0 : 정상) |

포장 라인과 공급업체 예는 집단마다 검사 수를 정해 둔 검사 기록(건수)을 직접 입력해 쓴다.

**질문** : 여러 집단(라인 · 공급업체 · 제품 유형)의 불량률이나 고장률은 서로 같은가?

**실습 전제** : 관측끼리 서로 독립이라고 가정한다.

In [ ]:
# 데이터 확인 : 행·열 수, 제품 유형별 건수
print("행·열:", data.shape)                               # shape : (행 수, 열 수)
print(data["Type"].value_counts())                       # value_counts : 값마다 몇 건인지 센다

행·열: (10000, 14)
Type
L    6000
M    2997
H    1003
Name: count, dtype: int64


## 1. 동질성 검정

**동질성 검정**은 둘 이상의 집단(모집단)에서 한 범주형 변수의 비율이 같은지 검정한다. 집단마다 표본 크기를 **미리 정해** 뽑은 자료에 쓴다.

동질성 검정이 필요한 이유 : 동질성 검정은 두 집단 이상의 비율이 같은지 한 번에 비교한다. 집단이 두 개면 2×2 카이제곱 검정이 되고, 이것이 동질성 검정의 가장 단순한 경우다. 집단이 세 개 이상일 때 특히 쓸모가 있다. 라인이 세 개면 두 라인씩 2×2 검정을 세 번 반복할 수도 있지만, 여러 번 검정하면 적어도 한 번 잘못 기각할 위험이 커진다. 동질성 검정은 모든 집단을 한 번에 비교하므로 이 문제가 없다. 평균을 비교할 때 t검정을 반복하지 않고 일원분산분석을 쓰는 것과 같은 이유다.

**적용 사례**

| 분야 | 질문 | 자료를 모은 방식 |
|---|---|---|
| 품질 관리 | 포장 라인 A · B · C 의 불량률이 같은가 | 라인마다 검사 수를 정해 검사 |
| 구매 · 입고 검사 | 공급업체마다 부품 불량률이 같은가 | 업체마다 같은 수량을 샘플링 |
| 설비 보전 | 제품 유형별 고장률이 같은가 | 유형마다 기록 수를 정해 추출 |

**가설 형식**

- 귀무가설(H₀): 모든 집단의 모비율은 같다. (p_A = p_B = p_C)
- 대립가설(H₁): 적어도 한 집단의 모비율이 다르다.

| 구분 | 동질성 검정 |
|---|---|
|  질문 | 여러 집단의 비율이 서로 같은가 |
| 모집단과 범주형 변수 | 모집단 2개 이상 · 범주형 변수 1개 (예 : 라인별로 뽑아 불량 여부만 확인) |
| 표본추출 방식 | 집단마다 크기를 정해 추출 |
| 기대빈도 E | 행 합계 × 열 합계 ÷ 전체 |
| 함수 | `stats.chi2_contingency` |

| 함수 | 기능 |
|---|---|
| `stats.contingency.expected_freq(교차표)` | 검정 없이 기대빈도만 계산한다 |
| `stats.chi2_contingency(교차표)` | 동질성 검정. `.statistic`(χ²) · `.pvalue`. contingency table |

교차표에는 비율이 아니라 **건수**를 넣는다.

공식 문서 : [pandas.DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) · [scipy.stats.contingency.expected_freq](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.contingency.expected_freq.html) · [scipy.stats.chi2_contingency](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html)

### 1-1. 포장 라인 A · B · C 의 불량률

포장 라인 A · B · C 에서 각각 500 · 400 · 300개를 정해 검사했다. 불량은 A 35개, B 12개, C 18개였다.

1) 가설 설정
   - 귀무가설(H₀): 세 라인의 모집단 불량률은 같다. (p_A = p_B = p_C)
   - 대립가설(H₁): 적어도 한 라인의 모집단 불량률이 다르다.
2) 유의수준 : α = 0.05
3) 조건 확인 → 검정 : 기대빈도가 모두 5 이상인지 확인한 뒤 `stats.chi2_contingency`
4) 결과 도출 : 라인별 불량률과 불량 칸의 관측 vs 기대로 어느 라인이 차이를 만들었는지 보고 두 줄 결론을 쓴다

In [ ]:
# [예제] 라인 A · B · C 에서 각각 500 · 400 · 300개를 정해 검사했다. 세 라인의 불량률이 같은지 검정한다
# 귀무가설(H₀): 세 라인의 모집단 불량률은 같다. (p_A = p_B = p_C)
# 대립가설(H₁): 적어도 한 라인의 모집단 불량률이 다르다.
# 유의수준 α = 0.05

# 1) 교차표 만들기 : 라인별 불량 · 양품 건수
line_table = pd.DataFrame({"불량": [35, 12, 18], "양품": [465, 388, 282]}, index=["라인 A", "라인 B", "라인 C"])
#   { } 안의 "불량", "양품" 이 열 이름이고, 대괄호 안 숫자가 라인 A · B · C 순서의 건수다
print(line_table)
line_total = line_table.sum(axis=1)                         # axis=1 : 행마다 가로로 더한다 → 라인별 검사 수 500 · 400 · 300
print("불량률(%):", (line_table["불량"] / line_total * 100).round(1).tolist())

# 2) 조건 확인 : 기대빈도 (세 라인의 불량률이 같다면 기대되는 건수)
line_expected = stats.contingency.expected_freq(line_table)
line_expected_table = pd.DataFrame(line_expected, index=line_table.index, columns=line_table.columns)   # 행 · 열 이름을 붙인다
print(line_expected_table.round(1))
print("가장 작은 기대빈도:", round(line_expected.min(), 1))

# 3) 동질성 검정
line_result = stats.chi2_contingency(line_table)
print("χ²:", round(line_result.statistic, 2), "/ p값:", round(line_result.pvalue, 4),"\n\n 기대빈도표:", line_result.expected_freq )

# [결과]
# - 불량률 A 7.0% · B 3.0% · C 6.0%, 가장 작은 기대빈도 16.2 ≥ 5 → 조건을 만족한다
# - χ² = 7.21, p = 0.0272
# - 불량 칸 관측 vs 기대 : A 35 > 27.1, B 12 < 21.7, C 18 > 16.2 → B 는 기대보다 불량이 적고, A 는 기대보다 많다
# - 통계적 결론 : p = 0.0272 < 0.05 이므로 귀무가설을 기각한다
# - 사실적 결론 : 세 라인의 불량률이 모두 같다고 보기 어렵다. 표본에서 A 7.0%, B 3.0%로 가장 큰 차이는 4.0%p다 (A 500 · B 400 · C 300개)

      불량   양품
라인 A  35  465
라인 B  12  388
라인 C  18  282
불량률(%): [7.0, 3.0, 6.0]
        불량     양품
라인 A  27.1  472.9
라인 B  21.7  378.3
라인 C  16.2  283.8
가장 작은 기대빈도: 16.2
χ²: 7.21 / p값: 0.0272 

 기대빈도표: [[ 27.08333333 472.91666667]
 [ 21.66666667 378.33333333]
 [ 16.25       283.75      ]]


### 1-2. 기각한 뒤 : 두 집단씩 비교 (참고)

세 집단 이상에서 기각하면 "적어도 한 집단이 다르다"까지만 알 수 있다. 어느 집단끼리 다른지 보려면 두 집단씩 2×2 검정을 한다. 이때 비교를 여러 번 하므로 유의수준을 비교 횟수로 나눠 기준을 엄격하게 한다. 이 방법을 **Bonferroni 보정**이라고 한다.

- 비교 횟수 = 집단 수 × (집단 수 − 1) ÷ 2. 세 집단이면 3 × 2 ÷ 2 = 3번 (A–B, A–C, B–C)
- 보정한 유의수준 = 0.05 ÷ 비교 횟수. 세 집단이면 0.05 ÷ 3 ≈ 0.0167

In [ ]:
# [예제] 참고 : 라인을 두 개씩 골라 2×2 카이제곱 검정을 하고, 보정한 유의수준 0.05 ÷ 3(비교 횟수) 과 비교한다
# 쌍마다 귀무가설(H₀): 두 라인의 모집단 불량률은 같다. (예 : p_A = p_B)
# 쌍마다 대립가설(H₁): 두 라인의 모집단 불량률은 다르다. (예 : p_A ≠ p_B)
alpha_adj = 0.05 / 3                                       # 보정한 유의수준
print("보정한 유의수준:", round(alpha_adj, 4))

ab = stats.chi2_contingency(line_table.loc[["라인 A", "라인 B"]])   # .loc[] : 두 라인 행만 고른 2×2 표
ac = stats.chi2_contingency(line_table.loc[["라인 A", "라인 C"]])
bc = stats.chi2_contingency(line_table.loc[["라인 B", "라인 C"]])
print("A–B p값:", round(ab.pvalue, 4))
print("A–C p값:", round(ac.pvalue, 4))
print("B–C p값:", round(bc.pvalue, 4))

# [결과]
# - A–B p = 0.0114 < 0.0167, A–C p = 0.6864, B–C p = 0.0800 (≥ 0.0167)
# - 통계적 결론 : 보정한 유의수준 0.0167 에서 A–B 쌍만 귀무가설을 기각한다
# - 사실적 결론 : 라인 A 와 B 의 불량률만 다르다고 볼 통계적 근거가 있다. 표본에서 A 7.0%, B 3.0%였다




from statsmodels.stats.proportion import proportions_chisquare_allpairs


count = line_table["불량"].to_numpy()       # 라인별 불량 건수
nobs = line_table.sum(axis=1).to_numpy()    # 라인별 전체 검사 건수

# 모든 라인 쌍 비교 + Bonferroni 보정
posthoc = proportions_chisquare_allpairs(
    count, nobs, multitest_method="b"
)

print(posthoc.summary())



보정한 유의수준: 0.0167
A–B p값: 0.0114
A–C p값: 0.6864
B–C p값: 0.08
Corrected p-values using Bonferroni p-value correction

Pairs                      p-values
(np.int64(0), np.int64(1))  0.02207
(np.int64(0), np.int64(2))       1
(np.int64(1), np.int64(2))  0.1574
0: 라인 A
1: 라인 B
2: 라인 C
Corrected p-values using Holm p-value correction

Pairs                      p-values
(np.int64(0), np.int64(1))  0.02207
(np.int64(0), np.int64(2))  0.5819
(np.int64(1), np.int64(2))  0.1049


## 2. 실습 : 제품 유형별 고장률 · 공급업체 불량률 비교

.

| 함수 | 기능 |
|---|---|
| `표.sample(n=300, random_state=1)` | 무작위로 300행을 뽑는다. `random_state` 는 실행할 때마다 같은 행이 뽑히도록 고정하는 값이다 |
| `pd.concat([표1, 표2, 표3])` | 여러 표를 위아래로 이어 붙인다 |
| `pd.crosstab(행으로 쓸 열, 열로 쓸 열)` | 원자료에서 두 범주형 열의 조합별 건수를 센 교차표를 만든다 |

공식 문서 : [pandas.DataFrame.sample](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html) · [pandas.concat](https://pandas.pydata.org/docs/reference/api/pandas.concat.html) · [pandas.crosstab](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)

In [ ]:
# [예제] 자료 준비 : 제품 유형 L · M · H 에서 300건씩 뽑아 하나의 표로 붙인다
type_sample = pd.concat([data.loc[data["Type"] == "L"].sample(n=300, random_state=1),
                         data.loc[data["Type"] == "M"].sample(n=300, random_state=1),
                         data.loc[data["Type"] == "H"].sample(n=300, random_state=1)])
#   data.loc[data["Type"] == "L"] : L 유형 행만 고른다. == 는 '같은가'를 묻는 비교다
#   .sample(n=300, random_state=1) : 그중 300건을 무작위로 뽑는다. random_state 로 실행할 때마다 같은 300건이 뽑히게 고정한다
#   pd.concat([표1, 표2, 표3]) : 세 표를 위아래로 이어 붙인다
print("뽑은 건수:", len(type_sample))
print(type_sample["Type"].value_counts())

# [결과]
# - L · M · H 각 300건, 모두 900건이다

뽑은 건수: 900
Type
L    300
M    300
H    300
Name: count, dtype: int64


In [ ]:
# [실습]
# 상황 : 위에서 뽑은 유형별 300건으로 세 유형의 고장률이 같은지 검정한다. 포장 라인 예제와 같은 순서로 푼다.
#        귀무가설(H₀): 세 유형의 모집단 고장률은 같다. (p_L = p_M = p_H)
#        대립가설(H₁): 적어도 한 유형의 모집단 고장률이 다르다.
#        유의수준 α = 0.05
# type_sample 로 제품 유형 × 고장 여부 교차표를 만들고, 유형별 고장률, 가장 작은 기대빈도, 동질성 검정의 χ² · p값을 구해 통계적 결론과 사실적 결론을 쓰시오.



In [ ]:
# [실습]
# 상황 : 공급업체 A · B · C 에서 들어온 부품을 업체마다 300개씩 검사했다. 불량은 A 8개, B 22개, C 11개였다.
#        귀무가설(H₀): 세 업체 부품의 모집단 불량률은 같다. (p_A = p_B = p_C)
#        대립가설(H₁): 적어도 한 업체 부품의 모집단 불량률이 다르다.
#        유의수준 α = 0.05
# 업체별 불량 · 양품 교차표를 만들고, 불량률, 기대빈도 표와 가장 작은 기대빈도, 동질성 검정의 χ² · p값을 구하시오.



In [ ]:
# [실습]
# 상황 : 위 공급업체 비교 결과를 보고서에 두 줄로 정리한다.
# 아래 결론 문장의 빈칸을 채우시오.
#    통계적 결론 : p = ______ ___ 0.05 이므로 귀무가설을 ___________.
#    사실적 결론 : 세 업체 부품의 불량률이 모두 같다고 보기 어렵다. 표본에서 업체 ___ 의 불량률이 ___% 로 가장 높았고, 기대보다 불량이 많았다 (업체별 300개).



## 3. 정리 : 동질성 검정

**결과를 읽는 순서** : 교차표(건수) · 집단별 비율 → 기대빈도(모두 5 이상인가) → χ² · p값 → 관측 vs 기대로 어긋난 칸 → 두 줄 결론 → (필요하면) 두 집단씩 비교

| 항목 | 내용 |
|---|---|
| 질문 | 여러 집단의 비율이 서로 같은가 |
| 자료를 모은 방식 | 집단마다 표본 크기를 미리 정해 추출 |
| 함수 | `stats.chi2_contingency(교차표)` |
| 기대빈도 | 행 합계 × 열 합계 ÷ 전체 (모든 집단의 비율이 같다면 나올 건수) |
| 조건 | 관측 독립, 기대빈도 모두 5 이상 |

### 해석 시 유의점

| 항목 | 해석 기준 |
|---|---|
| 기각했을 때 | 적어도 한 집단의 모집단 비율이 다르다는 의미이다. 어느 집단끼리 다른지는 사후검정으로 확인한다. |
| 기각하지 못했을 때 | 비율이 같다고 증명한 것이 아니다. 고장처럼 드문 결과는 표본이 작으면 차이를 확인하기 어렵다 |


공식 문서 : [scipy.stats.chi2_contingency](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html)